# Datakontroll — siste artikler

Henter artikler fra SQLite og leser tilhørende tekst fra Obsidian-vaulten.
Juster parameterne i neste celle.

In [5]:
# ── Parametere ────────────────────────────────────────────────
ANTALL      = 10      # Antall siste artikler å vise
SØKETEKST   = ""       # Filtrer på tittel/URL (tom = ingen filter)
ELEMENT_ID  = None     # Hent én spesifikk artikkel (None = bruk ANTALL)

In [6]:
import sqlite3
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown

# Last .env fra repo-rot
repo_rot = Path("__file__").resolve().parent.parent
load_dotenv(repo_rot / ".env")

DB_STI    = repo_rot / os.getenv("DATABASE_STI", "data/monitor.db")
VAULT_ROT = Path(os.getenv("VAULT_ROT", ""))

print(f"Database : {DB_STI}  (finnes: {DB_STI.exists()})")
print(f"Vault    : {VAULT_ROT}  (finnes: {VAULT_ROT.exists()})")

Database : C:\Users\Audun\2026\claude\personalisert_monitor\data\monitor.db  (finnes: True)
Vault    : C:\Users\Audun\2026\claude\OBSIDIAN\monitor-evals  (finnes: True)


In [7]:
conn = sqlite3.connect(DB_STI)
conn.row_factory = sqlite3.Row

if ELEMENT_ID is not None:
    sql = "SELECT * FROM elementer WHERE id = ?"
    params = (ELEMENT_ID,)
elif SØKETEKST:
    sql = """
        SELECT * FROM elementer
        WHERE tittel LIKE ? OR url LIKE ?
        ORDER BY id DESC LIMIT ?
    """
    mønster = f"%{SØKETEKST}%"
    params = (mønster, mønster, ANTALL)
else:
    sql = "SELECT * FROM elementer ORDER BY id DESC LIMIT ?"
    params = (ANTALL,)

artikler = conn.execute(sql, params).fetchall()
print(f"Fant {len(artikler)} artikkel(er)")

Fant 3 artikkel(er)


In [8]:
for art in artikler:
    # ── Metadata ──────────────────────────────────────────────
    meta = dict(art)

    # Sammendrag (kan være tomt)
    sam = conn.execute(
        "SELECT tekst, prompt_versjon, opprettet FROM sammendrag WHERE element_id = ? ORDER BY id DESC LIMIT 1",
        (meta["id"],)
    ).fetchone()

    # Artikkeltekst fra vault
    tekst = "*(vault_sti er None)*"
    if meta["vault_sti"]:
        vault_fil = VAULT_ROT / meta["vault_sti"]
        if vault_fil.exists():
            tekst = vault_fil.read_text(encoding="utf-8")
        else:
            tekst = f"*(fil ikke funnet: {vault_fil})*"

    # ── Visning ───────────────────────────────────────────────
    display(Markdown(f"---\n## [{meta['tittel']}]({meta['url']})"))
    display(Markdown(
        f"| Felt | Verdi |\n|------|-------|\n"
        + "\n".join(
            f"| `{k}` | {v} |"
            for k, v in meta.items()
        )
    ))

    if sam:
        display(Markdown(f"**Sammendrag** *(prompt: {sam['prompt_versjon']}, {sam['opprettet']})*\n\n{sam['tekst']}"))
    else:
        display(Markdown("*Ingen sammendrag ennå*"))

    display(Markdown("### Artikkeltekst"))
    display(Markdown(tekst))

---
## [M.025 Nothing and everything, all at once](https://www.molekyl.io/p/m025-nothing-and-everything-all-at?hide_intro_popup=true)

| Felt | Verdi |
|------|-------|
| `id` | 78 |
| `kilde_id` | 2 |
| `guid` | f00c7293-786d-4cdf-951d-0531de8221cb |
| `url` | https://www.molekyl.io/p/m025-nothing-and-everything-all-at?hide_intro_popup=true |
| `tittel` | M.025 Nothing and everything, all at once |
| `publisert` | None |
| `hentet` | 2026-05-07T07:42:24.921604+00:00 |
| `vault_sti` | artikler/molekyl-io/f00c7293-m-025-nothing-and-everything-all-at-once.md |
| `dead_letter` | 0 |
| `bilder_json` | None |

**Sammendrag** *(prompt: v1, 2026-05-07T07:51:08.914811+00:00)*

Overskrift  
AI gjør det mulig å bygge systemer direkte rundt prosessene dine – produkt og prosess smelter sammen

Oppsummering  
De siste årene har AI gjort det teknisk mulig å bygge programvare og systemer raskere og billigere enn før. Dette endrer hvordan man kan designe og implementere løsninger: Prosessen i virksomheten kan nå styre hvordan systemet ser ut, i stedet for at prosessen må tilpasses ferdige produkter. I praksis viskes skillet mellom produkt (det du bygger) og prosess (hvordan du jobber) ut, fordi systemet kan genereres og tilpasses underveis mens det brukes.

Hva er nytt

- Produkt følger prosess: Tidligere måtte prosesser tilpasses standardprodukter eller plattformer, fordi det var dyrt og tidkrevende å spesialbygge programvare. Nå kan du designe arbeidsflyten først, og så la AI-genererte systemer tilpasse seg denne. Eksempel: Et undervisningsopplegg får et eget AI-basert veiledningssystem bygget kun for én workshop, der systemet veileder studentene i sanntid uten at det eksisterte på forhånd. For ingeniøren betyr dette at kravinnhenting og systemdesign kan skje parallelt, og at endringer i prosessen kan implementeres løpende uten store omskrivninger.
- Produkt og prosess smelter sammen: Med generativ AI kan systemet endres eller genereres mens brukeren jobber. Eksempel: Under en workshop bygges og tilpasses et verktøy fortløpende etter hvert som prosessen utvikler seg, slik at både prosess og produkt påvirker hverandre i sanntid. For implementasjon betyr dette at systemet må støtte kontinuerlig oppdatering, og at observerbarhet og kontrollpunkter må bygges for å håndtere raske endringer og uforutsette feilmodi.
- Programvare genereres ved bruk: Nye mønstre gjør at hele brukeropplevelsen eller til og med virtuelle miljøer kan genereres dynamisk, bilde for bilde eller side for side, mens brukeren navigerer. Eksempel: En nettside eller et 3D-miljø eksisterer ikke før brukeren klikker eller beveger seg – AI genererer neste steg i sanntid. Dette krever at systemet håndterer latens, ressursbruk og feilhåndtering på en annen måte enn tradisjonelle applikasjoner, og at evalueringsopplegget må fange opp dynamiske feil og kvalitetsendringer som oppstår underveis.

Hvorfor det er viktig  
For AI-ingeniører endrer dette hvordan systemer bør designes og driftes. Det blir mulig å iterere raskere, bygge tettere på brukernes faktiske behov, og redusere teknisk gjeld knyttet til rigid arkitektur. Samtidig øker behovet for gode kontrollpunkter, logging og observerbarhet, fordi systemet kan endre seg mens det er i bruk. Feilmodi blir mer dynamiske, og det krever nye evalueringsstrategier og robusthetstiltak.

Regulatorisk kontekst  
Når systemer genereres eller tilpasses løpende, må logging og dokumentasjon følge med i sanntid for å møte krav fra AI Act og ISO 42001, særlig hvis løsningen brukes i høyrisiko-kontekster eller behandler persondata. Kontrollpunkter og revisjonsspor må kunne vise hvilke prosesser og data som lå til grunn for hver enkelt systemgenerering eller beslutning, og virksomheten må vurdere hvordan dynamiske endringer påvirker etterlevelse og sporbarhet.

### Artikkeltekst

---
element_id: f00c7293-786d-4cdf-951d-0531de8221cb
url: https://www.molekyl.io/p/m025-nothing-and-everything-all-at?hide_intro_popup=true
kildetype: manuell
---

# M.025 Nothing and everything, all at once

### On the collapse of the product-process distinction

*You are reading Molekyl, finally unfinished ideas on strategy, creativity and technology. [Subscribe here](https://molekyl.substack.com/subscribe) to get new posts in your inbox.*

---

Three and a half years ago, ChatGPT launched and transformed AI from a topic for the few to a topic for the many.

Since then, AI has changed absolutely nothing. And it has changed everything. At the same time.

This certainly sounds like a hallucination, but it isn’t. Let me explain why.

## Nothing has changed

The promise, hype and investments around AI the last 3.5 years have been massive. So has adoption of tools like ChatGPT. Most of us got a new icon or two on our home screens, and many are even paying for access to better versions of the apps hiding behind those icons.

But if we look around and remain honest with ourselves: not much else has changed.

Most organizations look and feel pretty much the same as in 2022. The people, or at least the types of people, are the same. The processes are the same. The structures are mostly the same. And the products or services have developed along the same trajectory as we were on in ’22.

This is true where I work, and it’s true in the majority of other organizations.

If we zoom further out, society as a whole is also pretty similar. The players are mostly the same, the institutions are the same, and so are the larger social systems in which we operate.

Most things are remarkably similar today to what they were 3-4 years ago. Seemingly untouched by AI.

The fact that so little has changed from AI might seem like a mystery in light of the hype and investments. But the explanation is pretty straightforward: it takes time for people, processes and social systems to adapt to new products.

Processes have always followed products. When a new technology application arrives, whether it’s the internet, email or the telephone, most of us first look at the new with (healthy?) scepticism. Then we carefully try it out within our established workflows. Finally, if we are sufficiently convinced, we start to rethink and redesign those workflows around the capabilities of the new. And first then, will the real potential of the new start to reveal itself.

The issue is that most of us are in the early stages of this process. We either look at AI with scepticism, or slap Copilot on top of existing processes designed for a non-AI world. While we scratch our heads, wondering why the revolution hasn't kicked in yet.

In other words, nothing has changed because for most, the adapting and rethinking of our processes around these new capabilities just hasn’t happened yet.

## Everything has changed

While nothing has changed on the surface, things are fundamentally different below it. There everything has changed.

Everything in the sense that a range of key assumptions most of us take for granted have shifted. Assumptions like “knowledge is a scarce resource”, “competence is in limited supply”, and “knowledge work requires time, money and effort”, are challenged in area after area as we speak.

One area where this change has gone further and reached the surface is in software development. Where vibe engineering tools like Claude Code allows us to do in minutes or hours what would previously require days or weeks of work.

The efficiency gains of such advances are impressive, but the deeper and more interesting implication of these developments lie elsewhere: that the increased speed and lower costs associate with AI-work allows us to reverse the very direction of the mentioned product-process relationship.

### Product follows process?

The rapid building speed with AI implies that we no longer need to adapt our processes to products designed to work across organizations and users. Now, we can increasingly often instead design whatever process we think is best in a given context, and use AI to build and tailor products to that specific process.

Product can suddenly follow process to a much greater extent than ever before.

But more than being a potential, this reversal is here, right now. Let me give you a first-hand example.

At NHH I am in charge of the mandatory bachelor course in strategy. Every semester my 400-450 students have three full-day workshops, where they in each work on a complicated strategy case in groups over six hours. Normally a workshop is led by myself, supported by a team of 1 PhD student and 8 master students that help me supervise the 120 groups.

This spring I had to rethink the set-up, as I had a large external keynote on the day of the first workshop, and I had three student assistants fewer than usual. With the help of my colleague and AI-partner-in-crime Alexander Lundervold, we built a system that would guide and tutor the students through the entire full day workshop. While the professor in the course, me, was occupied elsewhere.

A multi-agent AI system built for one specific workshop-process, in one specific course, at one specific school. Product followed process.

The AI-workshop system is an example of how AI and the product-process reversal allows us to do things we already do differently. Suddenly a digital version of the professor could tutor the students, patiently, six hours straight. Compared to a few minutes per group in a regular workshop.

But the same logic also extends to things we don’t do today at all. When the time, money or effort required to do knowledge work drops towards zero, we can suddenly do all sorts of things that didn’t make any economic sense when the work had to be done by humans.

For example, we can now update our strategic analyses every month, week or even morning, instead of once a year. We can run personalized, interlinked workshops for thousands of people in large companies (which is what we’re doing at [OAO](https://www.oao.co/), the startup I co-founded). Or we can build a [digital version of the NHH rector that interviews 1000 people](https://www.nhh.no/en/nhh-bulletin/article-archive/2026/march/when-ai-became-rector-800-conversations-that-reshaped-nhhs-strategy/) to get input to the school’s new strategy.

Beyond opening a range of new opportunities, the product-process reversal also can have competitive implications. So far, we mostly discuss how the high speed and low costs of AI makes it easier to imitate existing products and harder to sustain competitive advantages over time. But as more and more companies start to design their own idiosyncratic processes around the new AI capabilities, and then build customer products into those processes, things start to change. Then the one-size-fits-all approach to software will be replaced by processes and technology applications that are more dissimilar across firms. And make any advantages harder to attack in the process.

All because AI now allows all of us to develop the products we need for whatever process we think is best.

### Process equals product?

That said, the product-process reversal isn’t the end of it. It’s more the beginning of something even bigger and stranger: a world where the distinction between product and process might collapse entirely.

And that too is in a sense already here. Let me give you another first hand example.

A few months ago, as part of NHHs strategy process, Alexander Lundervold, Lasse Lien and myself ran a full day strategy workshop with NHH’s top 45 leaders. Right before the workshop started, we got the idea to build a tailored piece of software to elevate this particular workshop. So we did. Alexander spun up his coding agents, we refined and added ideas as the participants worked in groups, used the product during the workshop, adapted the workshop process to the new product, adapted the product to the updated workshop process, the process to the product, and so on. Both the tool and the workshop worked like a charm, and we have never used it since.

The process was the product, and the product was the process. The distinction between the two had collapsed.

Weird indeed, but not really special. Because the same trend is also visible elsewhere. At the frontier, labs are experimenting with coding models so fast that when you hit a button on a website, it takes you to a page that didn’t exist until your click. It gets written the moment you press it. Anthropic demonstrated such a system with their retro looking “ [Imagine with Claude](https://www.youtube.com/watch?v=dGiqrsv530Y) ” last year. Google launched [a demo of their Gemini Flash model](https://aistudio.google.com/apps/bundled/flash_lite_browser?utm_source=x&utm_medium=social&utm_campaign=&utm_content=&showPreview=true&showAssistant=true) earlier this year to do much the same, to generate software as you consume it. And more recently, [Flipbook](https://flipbook.page/), a visual browser that generates new pages as pixels while you surf, also push developments in this direction.

But it doesn’t stop there. Developments of world models, like [Google DeepMind’s Genie 3](https://deepmind.google/models/genie/), point in the same direction. World models like Genie 3 generate interactive 3D environments from a text prompt, frame by frame, at 24 frames per second. When you navigate the 3D environment, the next frame you see is produced in response to where you move. Continuously. Your process of interacting with this world becomes the product. And the product is the process. A consistent world that is generated and regenerated as you explore it. Tomorrow, the same technology could be fuelling software where screens become pixels generated as you use it.

Different angles, but one common direction. Products, software and even full virtual worlds produced as they are used. The product becomes the process. The process becomes the product. The distinction we have taken for granted dissolves in front of us.

## All at once

Where does this leave us? Nothing and everything has indeed changed with AI. But more striking than any one development or example, is the fact that nothing and everything is happening at once. Two very different realities, running in parallel, as we speak.

Most are still stuck in the first dimension, where AI is mostly a new icon on the home screen, that we try to wrap our heads around.

The other dimension is emptier, [weirder, and far far speedier](https://www.molekyl.io/p/m020-that-feeling). But its [still very real](https://www.molekyl.io/p/m021-the-gaze). And its open to both visitors and new residents. The only requirement to see it is that you behave as if some key assumptions underlying your work have been falsified by AI. Imagine what you might build if you put all the capable, but currently idle digital knowledge workers in your office to real work. Explore what process you would design if you don’t have to wait for someone else to build a product first.

If you do, you will likely realize that in this new dimension the key constraint is not capacity, as in the other dimension. The key constraints are ideas and imagination. It is knowing what to ask for. It is having problems worth solving and ideas worth pursuing.

But there are similarities too. The humans are the bottleneck also in the new. Just in a different way than before. Where progress is bound by their ideas and good problems, not by their capacity.

The two dimensions are still parallel, but likely won’t stay like that forever. At some point they will collapse into one. Just like the distinction between products and processes already. And then you will see it, whether you like it or not.


...

---
## [What is an agent harness in the context of large-language models?](https://parallel.ai/articles/what-is-an-agent-harness)

| Felt | Verdi |
|------|-------|
| `id` | 51 |
| `kilde_id` | 2 |
| `guid` | c59d0fdc-d4d4-4bf5-87fd-eaae241039f6 |
| `url` | https://parallel.ai/articles/what-is-an-agent-harness |
| `tittel` | What is an agent harness in the context of large-language models? |
| `publisert` | None |
| `hentet` | 2026-05-05T19:37:51.429384+00:00 |
| `vault_sti` | artikler/parallel-ai/c59d0fdc-what-is-an-agent-harness-in-the-context-of-large-language-models.md |
| `dead_letter` | 0 |
| `bilder_json` | None |

**Sammendrag** *(prompt: v1, 2026-05-05T19:39:07.818268+00:00)*

Overskrift  
Agent harness: Arkitekturmønsteret som gir AI-agenter vedvarende minne, verktøystøtte og kontroll

Oppsummering  
Agent harness er et arkitekturmønster som omgir en stor språkmodell med programvare som håndterer alt utenfor selve modellen: minne, verktøybruk, arbeidsflyt og tilstand. Dette mønsteret har vokst frem fordi rene språkmodeller ikke kan huske tidligere interaksjoner, bruke eksterne verktøy eller håndtere komplekse oppgaver som går over flere steg og økter. Med en agent harness kan du bygge AI-systemer som faktisk utfører oppgaver over tid, bruker eksterne ressurser og gir bedre kontroll på hva som skjer i hvert steg.

Hva er nytt

- Vedvarende minne og tilstandshåndtering  
  Problemet: LLM-er starter hver økt uten minne om tidligere samtaler eller oppgaver, og standard praksis har vært å sende med samtalehistorikk i prompten. Med en harness kan du implementere minne på tre nivåer: arbeidskontekst (det som sendes til modellen nå), sesjonstilstand (logg over hva som har skjedd i denne oppgaven), og langtidsminne (fakta eller erfaringer som lagres på tvers av oppgaver). I praksis betyr dette at en agent kan hente frem tidligere resultater, huske beslutninger, og fortsette der den slapp – for eksempel ved å lese en fremdriftslogg eller hente relevante fakta fra en kunnskapsbase. Dette gjør det mulig å bygge systemer som løser oppgaver over flere økter uten å miste tråden.

- Integrasjon med eksterne verktøy og handlinger  
  Problemet: LLM-er kan kun produsere tekst, men mange oppgaver krever handlinger som å søke på nett, kjøre kode eller oppdatere databaser. Harnessen overvåker modellens utdata for spesielle kommandoer (for eksempel search("...") eller python(...)), utfører handlingen i omverdenen, og mater resultatet tilbake til modellen. Et konkret scenario: En bruker ber agenten om å analysere ferske data. Modellen genererer en tool-call, harnessen henter dataene, og modellen får svaret som del av neste prompt. Dette gir agenten evne til å utføre reelle handlinger og jobbe med oppdaterte data, uten at modellen selv trenger å vite hvordan verktøyene virker.

- Strukturert arbeidsflyt, planlegging og verifikasjon  
  Problemet: Uten struktur kan AI-agenter levere overfladiske eller feilaktige resultater, særlig ved komplekse oppgaver. Harnessen kan tvinge frem en arbeidsflyt der agenten først lager en plan, så løser deloppgaver stegvis, og til slutt verifiserer resultatene. For eksempel: Ved kodegenerering kan harnessen sørge for at modellen skriver kode, kjører tester, og retter feil i en iterativ løkke – alt uten menneskelig inngripen. Harnessen kan også validere at utdata følger riktig format eller faktisk løser oppgaven, og be modellen om å forbedre svaret hvis noe mangler. Dette gir bedre kontroll på kvalitet og gjør feil lettere å oppdage og rette.

- Modulerbarhet og utvidbarhet  
  Problemet: Systemer som vokser blir fort uoversiktlige og vanskelige å endre. Moderne harness-design deler opp funksjonalitet i moduler (for eksempel minne, verktøybruk, planlegging), slik at du kan bytte ut eller utvide deler uten å endre hele systemet. I praksis kan du for eksempel legge til støtte for et nytt verktøy eller endre hvordan minnet håndteres, uten å skrive om resten av agenten. Dette reduserer teknisk gjeld og gjør systemet lettere å vedlikeholde og utvide.

Hvorfor det er viktig  
Agent harness gir deg kontrollpunkter og observerbarhet i AI-systemer som ellers ville vært svarte bokser. Du får mulighet til å logge, validere og styre agentens handlinger, og kan bygge systemer som faktisk holder seg stabile og forståelige under reell belastning. Det blir enklere å feilsøke, iterere og sikre kvalitet på tvers av komplekse arbeidsflyter. Harness-mønsteret gir også en vei til å bygge AI-produkter som kan håndtere lange prosesser, samarbeide med andre systemer, og levere pålitelig ytelse over tid.

Regulatorisk kontekst  
Agent harness gjør det mulig å implementere logging, minnehåndtering og kontrollpunkter som støtter krav fra regelverk som AI Act og ISO 42001. For

### Artikkeltekst

---
element_id: c59d0fdc-d4d4-4bf5-87fd-eaae241039f6
url: https://parallel.ai/articles/what-is-an-agent-harness
kildetype: manuell
---

# What is an agent harness in the context of large-language models?

## What is an agent harness?

AI agents today are more than just standalone models that take in and output text tokens. They operate within an ecosystem of tools, memory stores, and orchestrated workflows that enable them to perform complex tasks. In this context, a new term has emerged in the AI lexicon: the "harness."

[Industry Terms](https://parallel.ai/blog?tag=industry-terms)

23 min

## What is an agent harness?

In simple terms, an agent harness is the software infrastructure that wraps around a large language model (LLM) or AI agent, handling everything *except* the model itself. One AI architect defines an agent harness as “the complete architectural system surrounding an LLM that manages the lifecycle of context: from intent capture through specification, compilation, execution, verification, and persistence”, In practical terms, the harness is what connects an AI model to the outside world, enabling it to use tools, remember information between steps, and interact with complex environments.

This concept of a harness is relatively new as of the writing of this article. It arrived as developers noticed that the quality of an agent often depends not only on the underlying model’s intelligence, but also on how well the surrounding system supports that model with context. For example, early chatbot products like the original ChatGPT were just an LLM with a chat interface. Today’s advanced AI assistants have an entire stack: typically an orchestrator controlling multi-step reasoning, plus a harness that empowers the model to call tools, manage files, and handle long conversations. Together, the orchestrator and harness often determine the real-world effectiveness of the AI far more than incremental gains in model size or training data.

## Why did harnesses emerge in AI?

Harnesses emerged to solve practical challenges as AI agents took on more complex, long-running, and tool-oriented tasks. Modern AI agents are asked to do things that go beyond a single prompt-response exchange. For instance, writing software projects over multiple sessions, querying databases or analyzing large documents, or interacting with a user interface. These demands revealed several gaps that the core LLM alone could not fill:

- **Limited memory and context:** Standard LLMs have fixed context windows and start each session with no memory of previous interactions. It’s like an engineer with severe amnesia starting fresh each day. Harnesses address this by implementing (persistent context logs, summaries, or external knowledge stores) that carry information across sessions. For example, Anthropic’s Claude Agent SDK, described as a *general-purpose agent harness*, uses strategies like compaction (summarizing or condensing past interactions) to allow progress on tasks spanning many context windows.
- **Tool use and external actions:** LLMs by themselves can only produce text. But many tasks require actions like web search or browsing, code execution, database queries, or image generation. The harness bridges this gap by watching the model’s output for special *tool-call commands* and then executing those tools on the model’s behalf. In effect, the harness gives the model hands and eyes, turning textual intentions into real actions.
- **Structured workflows and planning:** Complex projects often need to be broken into subtasks with planning and verification at each step. A harness can enforce a disciplined approach, capturing the user’s intent, devising a plan or sequence of steps, and setting acceptance criteria for the outcome. Without structure, AI agents can produce superficially plausible results that fall apart on closer inspection. Harnesses emerged as a way to formalize planning and guardrails so that the agent’s output is actually useful and correct.
- **Long-horizon task management:** Especially for *long-running agents* (tasks that might span hours or days), harnesses provide a way to maintain state and continuity. A recent engineering blog from Anthropic noted that even very capable coding models would fail to build a large app without an external system to initialize the project, incrementally track progress, and leave behind artifacts (like a progress log or updated code) for the next session. The harness concept thus arose from the need to bridge the gap between sessions and ensure the agent makes consistent forward progress.

In summary, harnesses became necessary as AI moved from one-shot interactions to persistent, tool-using, multi-step autonomy. They address the “glue” issues – memory beyond the context window, interfacing with external systems, structuring multi-step work – that pure LLMs alone weren’t designed to handle.

## How does an agent harness work?

An agent harness typically works by intercepting and augmenting the communication between the user, the AI model, and any external tools or environments. Here’s a high-level look at how a harness operates within an AI agent system:

1. **Intent capture & orchestration:** First, the user’s request or high-level goal is captured. Often an *orchestrator* (another component of the system) will break this goal into sub-tasks or decide on a sequence of actions the AI should take. The harness works closely with this orchestrator by providing it the means to execute those actions. For example, the orchestrator might prompt the model for a plan or next step; the harness then ensures the model gets any needed context or tools at that step.
2. **Tool call execution:** As the model processes a task, it may output a special token or structured text indicating a tool use (e.g. search("climate change data") or python(code)). The harness monitors the model’s outputs and recognizes these tool calls. When a tool call is detected, the harness pauses the model’s text generation, executes the requested operation in the outside world (like performing the search or running the code in a sandbox), and then feeds the result back into the model’s context as if the model had “written” that result itself. This allows the model to reason over live data and outcomes. Essentially, the harness acts as the model’s proxy agent, turning the model’s intentions into actions and returning the observations.
3. **Context management & memory:** Throughout the interaction, the harness manages what information is given to the model. It may store a persistent task log or memory of what’s happened so far, separate from the transient prompt given to the model. Before each new model invocation (each “turn” or each new context window), the harness compiles a working context: a curated prompt that includes relevant history, essential facts, and recent results. Older or irrelevant information might be summarized or omitted to stay within token limits, a practice known as *context compaction* or *summarization*. The harness thus ensures the model always has the right information at the right time, avoiding issues like context window overflow or.
4. **Result verification & iteration:** A sophisticated harness doesn’t just execute tools blindly. It can also check the outputs. Some harnesses implement verification steps, such as checking that the format of the model’s output meets certain criteria or even running test cases on code the model wrote. If something is off, the harness might prompt the model to fix the issue in the next iteration. Harnesses designed for coding agents, for example, can include a cycle of *“write code -> run tests -> fix errors”* all orchestrated without human intervention. Moreover, harnesses often encourage incremental progress: they prompt the model to tackle one subtask at a time and save state (e.g., commit code to a repository or update a progress file) before moving on. This disciplined loop prevents the AI from trying to do too much at once and failing, a common issue in early agent experiments.
5. **Completion and handoff:** When the AI has completed the task (or a session times out), the harness handles the end-of-session routines. This might include saving artifacts (files created, summaries of work, a progress.txt log, etc.) that the next run can load in. In a way, the harness ensures that even if the AI agent stops and a new instance starts later (with no memory in the raw LLM), the *project itself has memory* via files and logs. This is crucial for long-running projects that the harness manages over multiple sessions.

Through all these stages, the harness remains **invisible to the end-user** but is crucial for the agent’s performance. Notably, a harness does **not** alter the LLM’s internal weights or training; it’s part of the *software architecture* around the model, not a retraining of the model itself. This means a harness can take a pre-trained model and by giving it the right support structure.

## Key components and features of agent harnesses

While implementations vary, most AI harnesses include a common set of components or features:

- **Tool integration layer:** At the heart of a harness is the ability to connect the model to external tools and APIs. This could include web search APIs like Parallel’s, database queries, calculators, code execution environments, image generators, or any custom tools. The harness defines a protocol for the model to request a tool (often via a special formatted output or function call syntax), and it handles executing that tool and feeding back results. A modern harness often comes with a suite of default tools (e.g., file read/write, web search, code interpreter) available to the model. For instance, the DeepAgents harness by provides a set of built-in tool calls and even a virtual file system “out of the box,” so the agent can read/write files or plan tasks without extra setup.
- **Memory and state management:** Harnesses implement memory beyond a single context window. This can include short-term memory (tracking the conversation or task state during a session) and long-term memory (persisting information across sessions). Some harness designs explicitly separate working context vs. session state vs. long-term memory. For example, *working context* is the immediate prompt given to the model (ephemeral); *session state* might be a durable log of what’s been done in the current task (persisted, but reset when the task is over); and *long-term memory* might be a knowledge base or vector store that persists across tasks or time (for general knowledge the agent has learned). By structuring memory this way, the harness can efficiently update just the necessary parts and avoid flooding the model with too much data each turn. Memory components often include summarization or retrieval: older interactions get distilled, and relevant facts are fetched when needed (similar to how a human might scan their notes before continuing a project).
- **Context engineering & prompt management:** Feeding the right prompt to the model is a science in itself. Harnesses perform context engineering – deciding what information to include or exclude at each model call. This involves techniques like *context isolation* (keeping different subtasks separate so they don’t confuse each other), *context reduction* (dropping or compressing irrelevant info to avoid context rot), and *context retrieval* (injecting fresh info such as documentation or search results at the right time). The harness may have modules that dynamically retrieve documents (RAG systems), or that rewrite the prompt for the first run versus subsequent runs (Anthropic describes using “a different prompt for the very first context window” in their harness structure to initialize things properly). All of this falls under the harness’s responsibility, ensuring the model is *prompted optimally* at each step.
- **Planning and decomposition:** Especially for agentic AIs (those that plan and act towards a goal), harnesses often include a planner or controller. This could be as simple as a predefined sequence of steps (for a narrow domain) or a more dynamic planner that uses the model to outline a strategy. Some harnesses prompt the model to produce a high-level plan which the harness then executes step by step, while others have hardcoded routines for things like “first do X, then do Y.” The key is that the harness can *guide the model* to avoid the one-shot, all-at-once failure mode. For example, Anthropic’s approach for long coding tasks involves an initializer agent (first-run harness prompt that sets up a project structure and task list) and then a coding agent that implements one feature at a time, guided by that structure. The harness enforces that incremental approach by the way it prompts and by how it checks off tasks after each session.
- **Verification and guardrails:** A robust harness will catch and correct errors. This can include schema or format validation (ensuring the model’s output can be parsed or meets a required format), logic checks (verifying the solution actually solves the problem or passes tests), and safety filters (preventing disallowed actions or content). For coding agents, a harness might run unit tests on generated code and only proceed if they pass. For a research assistant agent, the harness might verify that sources cited actually support the claims. These guardrails are part of the harness’s job to ensure quality and reliability of the AI’s actions, rather than leaving everything to the model’s own devices. As, simply adding more AI agents (like a separate “QA agent”) can backfire; often it’s better for the harness to make the primary agent *“be smart about doing its own QA”* and only escalate or reset when necessary.
- **Modularity and extensibility:** Many modern harness designs are modular, meaning you can plug in or toggle components. For example, an academic paper on *modular harnesses* for game-playing agents described a harness composed of distinct, each of which could be enabled or disabled to see its effect. The perception module converted visual game screens to text for the model, the memory module stored trajectories and reflections, and the reasoning module integrated everything in the model’s decision-making. Such modular harnesses let developers extend an agent’s abilities systematically. In general, a harness can be seen as a framework with “batteries included”, often coming with default modules for common needs (vision, code exec, web access, etc.) that can be refined or replaced as needed. This makes harnesses a higher-level construct than basic AI frameworks; they are more opinionated and feature-complete by design.

## Real-world examples of AI harnesses

Harnesses aren’t just theoretical. Many prominent AI platforms and research projects illustrate the harness concept in action:

- **Anthropic’s Claude Agent SDK:** Anthropic refers to its Claude Agent SDK as a *“general-purpose agent harness”* that is adept at coding and other tool-using tasks. It provides built-in context management (like automatic compaction of conversation history) and tool use capabilities to let Claude function as a long-running coding assistant. In their *Effective harnesses for long-running agents* report, Anthropic engineers described how they augmented this harness with an initializer/coding-agent pattern to keep Claude working coherently on projects that exceed its context window. Claude’s harness is what enables features such as writing and executing code, searching the internal knowledge base, and maintaining a claude-progress.txt log for handoff between sessions.
- **LangChain’s DeepAgents:** The LangChain library, known for its AI agent framework, introduced **DeepAgents** as an “agent harness” built on top of their ecosystem. Whereas LangChain provides abstractions (agents, tools, memory, etc.) and LangGraph handles execution and persistence (as an agent runtime), DeepAgents comes with **default prompts, tool handling, planning utilities, file system access, and more** baked in. The LangChain team likens DeepAgents to a general-purpose version of Claude’s harness (Claude Code) – basically a ready-to-go harness that developers can use for various purposes without assembling all pieces from scratch. This underscores how the term *harness* is used in the industry: DeepAgents isn’t a new model or just an SDK, but a *complete agent system* that wraps around models with lots of pre-configured capabilities.
- **Modular gaming agent harness:** In academic research, the paper *“General Modular Harness for LLM Agents in Multi-Turn Gaming Environments”* (ICML 2025) demonstrated a harness that allowed a single LLM to play diverse games by plugging in modules. Their **harness included perception, memory, and reasoning modules** attached to a GPT-4-class model, enabling it to see the game state, remember past moves, and deliberate effectively. The harness interfaced with the Gymnasium game API, feeding observations to the model and actions back to the game loop. Notably, this harness improved win rates across all tested games compared to an unharnessed baseline model, proving that a thoughtfully designed harness can significantly boost performance without changing the model itself. This is a clear validation that harnesses are effective: the model with a harness *consistently outperformed the same model alone*, because the harness gave it “hands” (to act in the game) and “memory” (to remember strategy) that it otherwise lacked.
- **Agentic application harnesses:** Beyond these, many AI applications have implicitly used harnesses even before the term was popular. AutoGPT and similar autonomous agents, for example, cobbled together loops of tool usage and memory – essentially a rudimentary harness – to let GPT-4 execute multi-step tasks. Microsoft’s Copilot chat for Office has an orchestrator and likely a harness that manages things like calling Bing search or inserting an image when the model asks for it. The recent flurry of “AI co-pilots” for coding (GitHub Copilot X, Cursor, etc.) all include sandboxed code execution harnesses so the AI can test code it writes. The industry is now recognizing these patterns and giving them a name (hence *“”* is becoming a discipline of its own).

## Harness vs. orchestration vs. framework: Clarifying the stack

It’s useful to distinguish an AI harness from related concepts like *agent frameworks* and *orchestrators*, since these terms can overlap:

- An **Agent framework** (such as LangChain, LlamaIndex, etc.) provides building blocks to create AI agents – things like abstractions for tools, memory, and chains of prompts. Think of frameworks as the libraries for constructing an agent. By contrast, an **Agent harness** is more of a full **runtime system with opinionated defaults and integrations**. In fact, a harness often *uses* a framework (for instance, DeepAgents harness uses LangChain). The harness is what you get when you assemble the pieces into a functioning whole.
- An **Orchestrator** in AI typically refers to the component that decides *when and how to call the model*, possibly multiple times, to accomplish a task. It might implement a reasoning loop (e.g., ReAct or tree-of-thought prompting) by parsing the model’s chain-of-thought and determining the next prompt. The orchestrator is about *logic and control flow*. The **harness**, on the other hand, is about *capabilities and side-effects*. It gives the model tools and manages input/output behind the scenes. They work together: the orchestrator might say “invoke the model with this prompt” or “loop again for another step”, and the harness ensures that when the model is invoked, it has the tools, context, and environment to do what’s asked. In short, orchestration is the brain of the operation, harness is the hands and infrastructure. Both are critical for complex AI agents, and improvements in either can dramatically improve an AI’s real-world performance.
- A **test harness** (an older term from software engineering) shouldn’t be confused with an AI or agent harness. A test harness is a framework for testing software, providing inputs and checking outputs automatically. While there is overlap (some AI harnesses include testing capabilities for code output), the term *harness* in the AI agent context is much broader. It’s not just for testing the model, but for empowering and managing the model’s operation. You might encounter phrases like “evaluation harness” in ML, for example, is a tool to measure model performance on benchmarks. That usage is context-specific. Unless “test” or “evaluation” is specified, “harness” in modern AI usually means an agent harness, the kind of runtime we’ve been discussing.

## Benefits of a well-designed harness

Harness engineering is quickly proving to be as important as model engineering. A well-designed harness can dramatically improve an AI system’s effectiveness, efficiency, and safety:

- **Higher task success rates:** By giving the model access to relevant tools and information, harnesses help the AI solve tasks it otherwise couldn’t. Experiments show that models achieve significantly better results when operating with a harness. For example, an AI playing a strategy game with a memory+perception harness won more games than the same AI without one. In coding, an AI with a harness that runs and debugs its code can complete programming tasks that a standalone LLM would fail due to runtime errors. The harness essentially *compensates for the model’s weaknesses* – be it lack of persistence, inability to use external knowledge, or propensity to make mistakes – leading to higher overall success.
- **Consistency on long tasks:** Harnesses shine in maintaining continuity. They prevent the agent from “forgetting” what it was doing after an interruption or context limit. By storing state and enforcing incremental progress, harnesses ensure that even if an agent must start fresh (new context), it can quickly reload what it needs and resume work. This addresses a major failure mode for long-running agents: without a harness, agents would either give up too early or repeat work aimlessly when faced with breaks in their context. A good harness, however, guides the agent to methodically carry on until completion, much like a project manager reminding a team what the next steps are after each meeting.
- **Better use of resources:** Harnesses can make AI systems more *efficient*. By structuring tool calls and context, a harness can reduce wasted tokens and unnecessary model calls. One approach described in harness design is to move some reasoning outside the model (e.g., using a knowledge graph or database for storing facts), which can “yield a 10-100x token reduction” in prompts – the model only gets the precise info it needs rather than huge swaths of text. This means cheaper and faster runs. Additionally, harnesses can cancel or correct wrong paths quickly (via verification), saving the model from spending a lot of tokens on a flawed approach.
- **Enhanced capabilities (without retraining):** Perhaps the biggest benefit is that harnesses extend what your AI can do without having to train a new model. Want your LLM to handle images? Put it in a harness that includes a vision module or an image captioning API. Need it to do math or complex logic? Give it a harness with a Python execution tool (like OpenAI’s Code Interpreter, which is essentially a harness feature). Historically, to add such capabilities you’d need to build a special model or fine-tune one; now, harnesses let a single general model perform a wide array of tasks by serving as the adapter between the model and specialized tools. This flexibility is a huge advantage, allowing organizations to leverage powerful pre-trained models in customized ways for their specific needs.
- **Improved reliability and safety:** By imposing structure and checks, a harness can reduce the AI’s tendency to go off track or produce harmful outputs. For example, if the model attempts an unsafe action or a disallowed content generation, the harness can have filters to catch that and stop or modify the request. It can also ensure the agent follows certain procedures (e.g. always cite sources for answers, or always get user confirmation before performing an irreversible action). These guardrails are easier to manage in the harness layer than baking everything into the model’s prompt, and they can be updated independently as new best practices emerge. In a sense, the harness is like the **governor on an engine**, preventing unwanted behavior while allowing productive work to continue.

It’s often said in AI product development now that **“the harness makes or breaks an AI product”**. Two products might use the same underlying LLM, but the one with a superior harness – offering better tool support, memory, and user guidance – will deliver a far better user experience. This is why companies like Anthropic, OpenAI, and others are investing heavily in harness engineering for their agents, and why we see new open-source harness projects emerging to help developers get this right.

## FAQs about agent harnesses

**Is an AI harness the same as prompt engineering?**

Not exactly. Prompt engineering is about crafting the text input to get the best response from a model. An AI harness includes prompt engineering as one of its duties (deciding what to feed the model), but goes much further – it manages tools, memory, and the whole loop of interactions. Think of prompt engineering as a technique that a harness might use. The harness itself is a larger architecture encompassing prompts, tool execution, result handling, and so on.

**Do I always need a harness to use an LLM effectively?**

For simple tasks (like one-off Q&A or text generation), you might not need anything fancy – just the model and a prompt could suffice. But as soon as you want the AI to do something non-trivial (e.g., use external data, solve multi-step problems, remember context over time, etc.), a harness (even a minimal one) is extremely useful. Many existing applications implicitly use harnesses. If you’ve used ChatGPT’s Code Interpreter or a plugin, you’ve seen a harness in action – it let the model run code or fetch info. So, you might not “need” a harness for very basic uses of LLMs, but harness-like components become crucial as you scale up complexity.

**How is harness engineering different from traditional software engineering?**

In many ways, harness engineering borrows concepts from software engineering – modular design, state management, input/output handling, testing, etc. The difference is you’re engineering around a non-deterministic AI core. The harness has to expect that the model might say or do unexpected things, and be designed to handle that gracefully. There’s also a lot of focus on *prompt design, tool APIs, and managing AI-specific limitations* (like token limits or hallucinations) which traditional software doesn’t have. One could say harness engineering is a fusion of backend engineering, plus a bit of UX design (for how the AI interacts), plus ML know-how. It’s a new discipline, and best practices are still being worked out in real time.

**Can multiple models share the same harness?**

Yes, in fact, a benefit of decoupling the harness from the model is that you can switch to a new or better model without rewriting the whole system. For example, you might start with GPT-4 as the model in your harness. If a new model comes out with longer context or better reasoning, you could replace GPT-4 with that model, and the harness would continue to provide memory, tools, and structure around it. Some harness setups even use *multiple* models concurrently (e.g., a smaller model for simple tasks, a bigger one for complex steps – the harness can route between them, known as model routing). So, the harness is essentially model-agnostic. That said, the prompt formats or tool call syntax might differ slightly between models, so you’d configure those details, but the overall harness logic remains applicable across models.

**Are harnesses relevant only for text-based LLM agents?**

The concept started with LLMs and tool-using chatbots, but it’s broadly applicable to any AI agent that operates sequentially. For example, a robotics researcher could talk about a “harness” that connects a planning AI to a robot’s sensor and motor controls – it’s the same idea of an interface layer. In reinforcement learning, what we used to call the *environment and wrapper* is analogous to an agent harness. So while the buzz is around harnesses for chatbots and coding assistants right now, the pattern of an external system enabling an AI to act will likely apply to many domains (vision systems, game AIs, autonomous vehicles, etc.). It’s a general principle: powerful AI brains need a body and tools – the harness is how we build that body in software.
